In [1]:
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google_auth_httplib2 import AuthorizedHttp
from googleapiclient.errors import HttpError
import httplib2
import time
import os
import io

/home/acomajuncosa/miniconda3/envs/adda4tb/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
def download_file(outfile):
    root = "/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation"
    file = os.path.basename(outfile)
    service_file = os.path.join(root, "service.json")
    GDRIVE_FOLDER_ID = "1FBELagBf9hlKVgvkaZ8YF60jKRAmsHPo"
    folder_id = GDRIVE_FOLDER_ID
    creds = Credentials.from_service_account_file(service_file, scopes=["https://www.googleapis.com/auth/drive.readonly"])
    for attempt in range(10):
        try:
            http = httplib2.Http(timeout=600)
            authed_http = AuthorizedHttp(creds, http=http)
            service = build("drive", "v3", http=authed_http)
            query = f"name='{file}' and '{folder_id}' in parents and trashed=false"
            results = service.files().list(q=query, fields="files(id)", supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            break
        except (HttpError, OSError):
            time.sleep(5)
            if attempt == 9:
                raise
    files = results.get("files", [])
    if not files:
        raise FileNotFoundError(f"'{file}' not found in folder {folder_id}. Consider checking available chunks in data/chunks/chunks.csv")
    if len(files) > 1:
        raise RuntimeError(f"Multiple files named '{file}' are found...")
    file_id = files[0]["id"]
    request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(outfile, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request,chunksize=100 * 1024 * 1024)  # 100 MB chunks
        done = False
        retries = 0
        while not done:
            try:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Download {int(status.progress() * 100)}%\n")
            except (HttpError, OSError):
                retries += 1
                print(f"Error found when downloading file. Trying again...[{retries}/10]")
                if retries >= 10:
                    raise
                time.sleep(10)

def get_splits(path):
    splits = os.listdir(path)
    return [i.replace(".csv", "") for i in sorted(splits)]

root_gpu = "/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation"
PATH_TO_OUTPUT = os.path.join(root_gpu, "processed", "unidock_REAL_docking", "inference_10B")

# Get splits (994)
SPLITS = get_splits(os.path.join(PATH_TO_OUTPUT, 'A_proteins'))

In [ ]:
for split in SPLITS:

    PATH_TO_IDs = os.path.join(root_gpu, "tmp", f"{split}_SMILES_IDs.tsv.zip")

    # Download splits id file
    if os.path.exists(PATH_TO_IDs) == False:
        download_file(PATH_TO_IDs)

Download 100%

Download 100%

Download 97%

Download 100%

Download 96%

Download 100%

Download 96%

Download 100%

Download 96%

Download 100%

Download 96%

Download 100%

Download 95%

Download 100%

Download 93%

Download 100%

Download 93%

Download 100%

Download 94%

Download 100%

Download 94%

Download 100%

Download 94%

Download 100%

Download 98%

Download 100%

Download 100%

Download 92%

Download 100%

Download 100%

Download 100%

Download 98%

Download 100%

Download 98%

Download 100%

Download 99%

Download 100%

Download 96%

Download 100%

Download 95%

Download 100%

Download 97%

Download 100%

Download 100%

Download 98%

Download 100%

Download 93%

Download 100%

Download 94%

Download 100%

Download 97%

Download 100%

Download 98%

Download 100%

Download 98%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Download 100%

Downl